# DSPy Optimizer Catalog — All 14 Optimizers

**Week 6 | Notebook 9 of 12**

**What you'll learn:**
- What `compile(program, trainset, metric)` actually does
- The four levers optimizers turn: demos, instructions, weights, ensembles
- All 14 optimizers from the [official API reference](https://dspy.ai/current/api/optimizers/):
  `LabeledFewShot`, `KNN`, `KNNFewShot`, `BootstrapFewShot`, `Ensemble`, `InferRules`,
  `BootstrapRS` / `BootstrapFewShotWithRandomSearch`, `COPRO`, `SIMBA`, `MIPROv2`, `GEPA`,
  `BootstrapFinetune`, `BetterTogether`
- Each optimizer's mechanism, LM-call budget, and when to reach for it
- Mermaid flow diagrams of the major optimization loops

**Runtime:** ~25 minutes

**Note:** Notebooks 2 and 5 go deep on `BootstrapFewShot`, `MIPROv2`, and `GEPA` — here they
appear as theory + diagrams + pointers so all 14 can be compared in one place. The four
expensive optimizers are shown but intentionally **not executed** (they either have dedicated
notebooks or launch real training jobs); the other ten run live with tiny budgets.

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/09_optimizer_catalog.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/09_optimizer_catalog.ipynb
Task:      Optimizer catalog — all 14 optimizers
Calls:     ~90

With GPT-4o:       $0.90 USD
With GPT-4o-mini:  $0.09 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup — One Task, Every Optimizer

In [2]:
import logging

import dspy
import tqdm as _tqdm

from src.config import get_dspy_lm, print_config
from src.datasets import generate_qa_pairs

# Optimizers log step-by-step INFO lines and render live tqdm progress bars
# (which embed \r control chars into saved outputs) — quiet both for readability.
logging.getLogger("dspy.teleprompt").setLevel(logging.WARNING)
logging.getLogger("dspy.evaluate").setLevel(logging.WARNING)
logging.getLogger("dspy.predict.predict").setLevel(logging.ERROR)


class _QuietTqdm(_tqdm.tqdm):
    def __init__(self, *args, **kwargs):
        kwargs["disable"] = True
        super().__init__(*args, **kwargs)


_tqdm.tqdm = _QuietTqdm  # delete this patch if you want to watch live progress

print_config()

lm = get_dspy_lm()
dspy.configure(lm=lm)

print(f"\n✅ DSPy configured with: {lm.model}")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      openai
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.6-flash
  groq model:      openai/gpt-oss-120b
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ DSPy configured with: openai/gpt-4o


Every demo in this notebook compiles the **same** task, so the optimizers can be
compared apples-to-apples. The data is synthetic (`generate_qa_pairs`, zero data cost) — the
same loader Notebook 2 uses.

In [3]:
qa_data = generate_qa_pairs(40)

examples = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question")
    for d in qa_data
]

split = int(len(examples) * 0.7)
trainset, devset = examples[:split], examples[split:]
train_tiny = trainset[:12]  # small subset keeps live demos cheap

print(
    f"Trainset: {len(trainset)} | Devset: {len(devset)} | Tiny trainset for demos: {len(train_tiny)}"
)

Trainset: 28 | Devset: 12 | Tiny trainset for demos: 12


In [4]:
def answer_metric(example, prediction, trace=None):
    """Word-overlap ratio: rewards predictions that cover the expected answer.

    Deliberately simple — a real metric might use an LLM judge. Note that every
    optimizer below consumes this *same* function; the optimizer is what changes.
    """
    expected = set(example.answer.lower().split())
    predicted = set(prediction.answer.lower().split())
    if not expected:
        return 0.0
    return 1.0 if len(expected & predicted) / len(expected) >= 0.5 else 0.0


class QA(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class SimpleQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(QA)

    def forward(self, question):
        return self.generate(question=question)


baseline = SimpleQA()
print("Baseline program ready.")

Baseline program ready.


## 2. How to Think About Optimizers

**Compilation** is DSPy's term for automatic program improvement: you hand
`optimizer.compile(student, trainset, metric)` an unoptimized program, a set of examples, and a
quality metric, and you get back a better program. *How* it gets better depends on which of the
four levers the optimizer turns:

1. **Demos** — which worked examples appear in the prompt (`LabeledFewShot`, `BootstrapFewShot`, `KNNFewShot`, …)
2. **Instructions** — the task text of each signature (`COPRO`, `MIPROv2`, `SIMBA`, `GEPA`, `InferRules`)
3. **Weights** — fine-tuning the LM itself (`BootstrapFinetune`, `BetterTogether`)
4. **Ensembles & retrieval** — combining programs or selecting demos per input (`Ensemble`, `KNN`)

```mermaid
flowchart LR
    P["your dspy.Module"] --> FS & PS & ENS & FT
    subgraph FS["Few-shot only — 0 optimization calls"]
        LFS["LabeledFewShot"]
        KNN["KNN / KNNFewShot"]
    end
    subgraph PS["Prompt search — LM calls"]
        BFS["BootstrapFewShot"]
        RS["BootstrapRS (= BootstrapFewShotWithRandomSearch)"]
        CP["COPRO"]
        SB["SIMBA"]
        MP["MIPROv2"]
        GP["GEPA"]
        IR["InferRules"]
    end
    subgraph ENS["Combine"]
        EN["Ensemble"]
    end
    subgraph FT["Fine-tune weights — training jobs"]
        BF["BootstrapFinetune"]
        BT["BetterTogether"]
    end
```

**How to read each entry:** *What it is* → *How it works* → *Key params* → *Budget* (approximate
LM calls for a small task) → *When to use / avoid*. Budgets scale with trainset size, search
breadth, and metric cost — treat the numbers as order-of-magnitude guides.

## 3. `LabeledFewShot` — Just Show It Labeled Examples

**What it is:** the simplest optimizer. No LM calls at all — it copies examples from
your trainset into each predictor's prompt as demonstrations.

**How it works:** `compile()` samples `k` examples per predictor and attaches them as `demos`.
The prompt now contains worked examples; the model imitates them. That's the whole mechanism —
which is why it's the baseline every other few-shot optimizer compares against.

**Key params:** `k` (demos per predictor).

**Budget:** 0 optimization calls (+ the normal inference cost of longer prompts).

**When to use:** you already have labeled data and want the cheapest possible boost. **Avoid
when** examples are long (context bloat) or when unlabeled data is all you have (use
`BootstrapFewShot` instead).

In [5]:
compiled = dspy.LabeledFewShot(k=4).compile(baseline, trainset=train_tiny, sample=False)

# ChainOfThought is a Module wrapping an inner Predict — demos attach to the Predict
demos = compiled.generate.predict.demos
print(f"Demos attached to the predictor: {len(demos)}")
for demo in demos[:2]:
    print(f"  Q: {demo.question}")
    print(f"  A: {demo.answer}")

Demos attached to the predictor: 4
  Q: What is DSPy?
  A: DSPy is a framework for programming language models.
  Q: What is Vector DB?
  A: Vector databases store embeddings for similarity search.


## 4. `KNN` — Retrieval Over Your Trainset

**What it is:** a k-nearest-neighbors retriever, not a prompt optimizer. It embeds
every trainset example once, then returns the `k` most similar examples for any input.

**How it works:** at setup, all trainset inputs are embedded and indexed in memory. At query
time, the input is embedded and scored against the index (cosine similarity via dot product).
The result is a *list of `dspy.Example`s* you can inspect — or feed to `KNNFewShot` (§5).

**Key params:** `k`, `trainset`, `vectorizer` (a `dspy.Embedder` — hosted or local, see
Notebook 8).

**Budget:** ~1 embedding call at setup (batched) + 1 per query. No generation calls.

**When to use:** per-input demo selection, retrieval demos, prototyping RAG. **Avoid when** the
trainset is huge and memory-bound without an ANN index.

In [6]:
vectorizer = dspy.Embedder("openai/text-embedding-3-small")  # any dspy.Embedder works

knn = dspy.KNN(k=2, trainset=train_tiny, vectorizer=vectorizer)
neighbors = knn(question="What is RAG?")

print("2 nearest examples to 'What is RAG?':")
for neighbor in neighbors:
    print(f"  {neighbor.question:<28} -> {neighbor.answer[:50]}")

2 nearest examples to 'What is RAG?':
  What is RAG?                 -> Retrieval-Augmented Generation combines search wit
  What is RAG?                 -> Retrieval-Augmented Generation combines search wit


## 5. `KNNFewShot` — Per-Input Few-Shot Selection

**What it is:** combines KNN retrieval with `BootstrapFewShot`. Instead of a fixed set
of demos for every input, each input gets the demos of its *own* nearest neighbors.

**How it works:** at inference time, `KNNFewShot` (1) retrieves the k nearest trainset examples
to the input, (2) runs `BootstrapFewShot` on just that neighborhood to build a mini few-shot
prompt, and (3) answers with it. The prompt adapts per query — coding questions get coding
demos, finance questions get finance demos.

**Key params:** `k`, `trainset`, `vectorizer`, plus any `BootstrapFewShot` args
(`max_bootstrapped_demos`, `max_labeled_demos`, `metric`).

**Budget:** KNN embeddings + a full `BootstrapFewShot` run **per inference call** — this is the
expensive part; use small `max_*_demos` or cache compiled neighborhoods in production.

**When to use:** heterogeneous trainsets where relevant demos differ per input. **Avoid when**
latency matters or the trainset is homogeneous (plain `BootstrapFewShot` is cheaper).

In [7]:
knn_few_shot = dspy.KNNFewShot(
    k=2,
    trainset=train_tiny,
    vectorizer=vectorizer,
    metric=answer_metric,
    max_bootstrapped_demos=2,
    max_labeled_demos=2,
)
compiled = knn_few_shot.compile(baseline)

result = compiled(question="What is DSPy?")
print(f"Answer: {result.answer}")
print("(demos were selected per-query from the nearest neighbors of this question)")

Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Answer: DSPy is a software library for simplifying the creation and deployment of machine learning applications.
(demos were selected per-query from the nearest neighbors of this question)


## 6. `BootstrapFewShot` — Self-Generated, Metric-Verified Demos

**What it is:** generates its own training demos by *running the program*, and keeps
only the runs that pass your metric.

**How it works:** the student (or a teacher program) attempts each trainset example; traces
that the metric scores as correct become bootstrapped demos. No labeled answers needed — the
metric is the judge. This is the workhorse few-shot optimizer and the building block inside
`BootstrapRS`, `KNNFewShot`, and `InferRules`.

**Key params:** `metric`, `metric_threshold` (min score to accept a trace), `teacher_settings`
(use a stronger LM to generate demos), `max_bootstrapped_demos` / `max_labeled_demos`,
`max_rounds`.

**Budget:** one inference call per trainset example (+ retries on failures) — ~10–20 calls on
a small trainset.

**When to use:** the default first optimizer for most tasks. **Avoid when** your metric is
noisy at single-example granularity. (Full demo with devset evaluation in Notebook 2.)

In [8]:
optimizer = dspy.BootstrapFewShot(
    metric=answer_metric, max_bootstrapped_demos=3, max_labeled_demos=3, max_rounds=1
)
compiled = optimizer.compile(baseline, trainset=train_tiny)

print(f"Bootstrapped demos kept: {len(compiled.generate.predict.demos)}")
result = compiled(question="What is Attention?")
print(f"Answer: {result.answer}")

Bootstrapped 3 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapped demos kept: 3
Answer: Attention is a mechanism in neural networks enabling models to focus on specific parts of the input data, enhancing performance in tasks like translation and summarization.


## 7. `Ensemble` — Vote Across Programs

**What it is:** combines several *already-compiled* programs into one predictor —
not an optimizer of prompts but of **redundancy**.

**How it works:** `compile(programs)` takes a list of programs. At inference, the ensemble runs
(all or a random `size` sample of) them and reduces their outputs with `reduce_fn` —
`dspy.majority` votes on shared output fields, or pass your own reducer (e.g. average numeric
fields, LLM-judge pick). Different members can come from different optimizers or seeds, making
errors decorrelate.

**Key params:** `reduce_fn` (e.g. `dspy.majority`), `size` (subsample per call), `deterministic`
(not yet implemented — must stay `False`).

**Budget:** the sum of member inference costs; compilation is free.

**When to use:** high-stakes predictions where you can pay 3–5× inference cost for accuracy.
**Avoid when** latency/cost budgets are tight.

In [9]:
member_a = dspy.LabeledFewShot(k=3).compile(baseline, trainset=train_tiny, sample=False)
member_b = dspy.LabeledFewShot(k=3).compile(baseline, trainset=train_tiny[3:], sample=False)

ensemble = dspy.Ensemble(reduce_fn=dspy.majority).compile([member_a, member_b])
result = ensemble(question="What is a Vector DB?")
print(f"Majority answer: {result.answer}")

Majority answer: A Vector DB is a database designed to store and manage vector data for efficient similarity searches.


## 8. `InferRules` — Turn Demos Into Natural-Language Rules

**What it is:** extends `BootstrapFewShot` — after bootstrapping demos, it asks an LLM
to *read the demos and write instructions*, then keeps whichever rules help on a validation
split.

**How it works:** (1) bootstrap demos as usual; (2) for each candidate round, a rules-induction
program summarizes patterns across the demos into natural-language rules; (3) the rules are
appended to the signature's instructions; (4) the candidate is evaluated on a valset and the
best-scoring candidate wins. Humans call this "writing the style guide from examples" — the
optimizer does it automatically.

**Key params:** `num_candidates` (induction rounds), `num_rules` (rules per round), plus all
`BootstrapFewShot` args (`metric`, `max_*_demos`).

**Budget:** bootstrap cost + `num_candidates` × (rule induction + valset evaluation) — tens of
calls on small data.

**When to use:** tasks with clear stylistic or formatting conventions spread across examples.
**Avoid when** a single canonical instruction already exists (use `COPRO`/`SIMBA` instead).

In [10]:
optimizer = dspy.InferRules(
    num_candidates=2,
    metric=answer_metric,
    max_bootstrapped_demos=2,
    max_labeled_demos=2,
    max_rounds=1,
)
compiled = optimizer.compile(baseline, trainset=train_tiny)

print("Instructions after rule induction:")
print(compiled.generate.predict.signature.instructions[:300])
result = compiled(question="What is Fine-tuning?")
print(f"\nAnswer: {result.answer}")

Bootstrapped 2 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.


Instructions after rule induction:
Answer questions with short factual responses.

Please adhere to the following rules when making your prediction:
1. Ensure the input field is formatted as a question beginning with "What is" followed by the technical term of interest.
2. The output field should provide a concise and accurate defini

Answer: Fine-tuning is the process of further training a pre-trained model on a specific task or dataset to improve its performance.


## 9. `BootstrapRS` / `BootstrapFewShotWithRandomSearch` — Search Over Bootstraps

**What it is:** random search over `BootstrapFewShot` configurations. Note: in the
installed dspy these two names are the **same class**
(`dspy.BootstrapRS` is an alias of `dspy.BootstrapFewShotWithRandomSearch`) — the docs list
both, the API has one.

**How it works:** run `BootstrapFewShot` several times with random demo configurations
(`num_candidate_programs` candidates), evaluate each on the trainset (or a valset), and return
the best-scoring compiled program. Random search sounds naive but is a strong baseline for
low-dimensional spaces like "which demos, how many."

**Key params:** `num_candidate_programs`, `metric`, `stop_at_score` (early stop when a
candidate is good enough), plus `BootstrapFewShot` args.

**Budget:** candidates × (bootstrap + evaluation) — the multiplier is the point; keep
`num_candidate_programs` small (3–8) until you know the metric is reliable.

**When to use:** squeezing accuracy out of few-shot prompting with spare budget. **Avoid when**
a single bootstrap already saturates your metric.

In [11]:
optimizer = dspy.BootstrapFewShotWithRandomSearch(
    metric=answer_metric,
    num_candidate_programs=2,
    max_bootstrapped_demos=2,
    max_labeled_demos=2,
    max_rounds=1,
)
compiled = optimizer.compile(baseline, trainset=train_tiny)

result = compiled(question="What is a Transformer?")
print(f"Best-of-search answer: {result.answer}")
print(f"Demos in winning program: {len(compiled.generate.predict.demos)}")

Going to sample between 1 and 2 traces per predictor.
Will attempt to bootstrap 2 candidate sets.


New best score: 41.67 for seed -3
Scores so far: [41.67]
Best score so far: 41.67
New best score: 66.67 for seed -2
Scores so far: [41.67, 66.67]
Best score so far: 66.67


Bootstrapped 2 full traces after 6 examples for up to 1 rounds, amounting to 6 attempts.
Scores so far: [41.67, 66.67, 41.67]
Best score so far: 66.67
Bootstrapped 2 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.


Scores so far: [41.67, 66.67, 41.67, 41.67]
Best score so far: 66.67
Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
New best score: 75.0 for seed 1
Scores so far: [41.67, 66.67, 41.67, 41.67, 75.0]
Best score so far: 75.0
5 candidate programs found.
Best-of-search answer: A Transformer is a neural network architecture that utilizes self-attention mechanisms for processing sequences of data, widely used in natural language processing tasks.
Demos in winning program: 2


## 10. `COPRO` — Hill-Climbing Over Instructions

**What it is:** *Content Programmer* — one of the earliest DSPy optimizers. Iteratively
**rewrites the signature's instructions** and keeps improvements, like coordinate ascent on
prompt text.

**How it works:** start from the current instructions; each round, an LLM proposes `breadth`
new instruction variants informed by previously *attempted* instructions and their scores;
evaluate each on the trainset; descend into the best (`depth` rounds). The prompt text itself
is the search space — demos are not the focus.

**Key params:** `metric`, `breadth` (proposals per round), `depth` (rounds), `init_temperature`
(diversity of proposals), `prompt_model` (a stronger model can write better instructions).

**Budget:** roughly breadth × depth × trainset evaluations + proposal calls — grows fast; keep
`breadth`/`depth` tiny (2–3) for demos.

**When to use:** tasks where wording matters (formatting, tone, structured extraction) and you
have no demos. **Avoid when** instructions are already well-tuned (use `SIMBA`/`GEPA`) or the
task is demo-driven.

In [12]:
optimizer = dspy.COPRO(metric=answer_metric, breadth=2, depth=1, init_temperature=1.0)
compiled = optimizer.compile(baseline, trainset=train_tiny[:8])

print("Optimized instructions:")
print(compiled.generate.predict.signature.instructions[:300])
result = compiled(question="What is Chain-of-Thought?")
print(f"\nAnswer: {result.answer}")

Optimized instructions:
Provide concise and factual definitions for technical terms based on a given question starting with "What is." Each response should deliver a single-sentence explanation that precisely captures the essence of the term in formal and field-relevant language. Ensure consistency by repeating exact quest

Answer: Chain-of-Thought is a reasoning technique used in AI models to produce intermediate steps during problem-solving to enhance complex decision-making processes.


## 11. `SIMBA` — Mini-Batch Self-Reflection

**What it is:** *Stochastic Introspective Mini-Batch Ascent* — the LLM analyzes its
own failures on mini-batches and writes rules for itself, one small step at a time.

**How it works:** sample a mini-batch of `bsize` examples; run the current program and find
examples with high output variability (the "hard" ones); ask the LM to reflect and produce
either self-reflective **rules** (added to instructions) or successful **demos**; evaluate the
candidate and keep it if the running average score improves. The `max_steps` loop repeats with
fresh mini-batches. Compared to `COPRO`'s global instruction rewrites, SIMBA's updates are
**local, data-driven, and batched** — often more sample-efficient.

**Key params:** `metric`, `bsize` (mini-batch size — must be ≤ trainset length), `num_candidates`,
`max_steps`, `max_demos`, `prompt_model`.

**Budget:** steps × (batch inference + reflection + candidate eval) — tens of calls with small
settings; the reflection calls are the distinctive extra cost.

**When to use:** mid-size trainsets (≥ ~32 examples at default settings) where failures cluster.
**Avoid when** the trainset is tiny (bootstrap noise dominates) or a valset exists and budget
allows `MIPROv2`/`GEPA`.

In [13]:
optimizer = dspy.SIMBA(metric=answer_metric, bsize=4, num_candidates=2, max_steps=2, max_demos=2)
compiled = optimizer.compile(baseline, trainset=train_tiny[:8])

result = compiled(question="What is Prompt Engineering?")
print(f"Answer: {result.answer}")

Answer: Prompt engineering is the process of designing and refining inputs to guide a language model to produce specific or improved outputs.


## 12. `MIPROv2` — Bayesian Optimization Over Instructions × Demos

**What it is:** the current default "serious" prompt optimizer. Searches the
**joint space of signature instructions and demo selections** with Bayesian optimization,
using a surrogate model to propose the next candidate to evaluate.

**How it works:** (1) seed a pool of instruction candidates and bootstrap candidate demos;
(2) build a surrogate model mapping (instruction, demos) → predicted score; (3) propose the
most promising untried combo; (4) evaluate, update the surrogate, repeat within budget;
(5) return the best program found. `auto="light"|"medium"|"heavy"` presets scale the budget.

```mermaid
flowchart TD
    A["Seed: instruction candidates + bootstrapped demos"] --> B["Surrogate model predicts scores for untried combos"]
    B --> C["Evaluate the most promising candidate on the trainset"]
    C --> D["Update surrogate with the real score"]
    D --> E{Budget left?}
    E -- yes --> B
    E -- no --> F["Best program found"]
```

**Key params:** `metric`, `auto` presets or `max_bootstrapped_demos`/`max_labeled_demos`,
`prompt_model` / `task_model`, `teacher_settings`.

**Budget:** tens to hundreds of calls — the strongest prompt-only optimizer, priced accordingly.

**When to use:** the go-to when `BootstrapFewShot` plateaus and you can afford a real search.
**Avoid when** you need the cheapest path. 🚧 *Not executed here — a full live demo is the
subject of **Notebook 2** (`02_optimizers.ipynb`).*

In [14]:
# 🚧 SHOWN FOR COMPLETENESS — NOT EXECUTED (see Notebook 2 for a full live run)
optimizer = dspy.MIPROv2(
    metric=answer_metric,
    auto="light",  # preset budget: "light" | "medium" | "heavy"
)
compiled = optimizer.compile(baseline, trainset=trainset)  # noqa: F841
# Evaluate with: dspy.Evaluate(devset=devset, metric=answer_metric)(compiled)

Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6
Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Bootstrapping set 4/6
Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Bootstrapping set 5/6


Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 6/6
Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.


## 13. `GEPA` — Reflective Evolution Over a Pareto Frontier

**What it is:** *Genetic-Pareto* optimizer — an evolutionary search where new
candidates are created not by mutation but by **LLM reflection on failure traces**, and the
population is kept on a quality-vs-cost Pareto frontier.

**How it works:** run a parent program on a mini-batch and collect full traces; a reflection
LLM reads the failing traces plus the metric and explains *why* the program lost points; that
feedback text is used to propose improved candidate programs (offspring); candidates are
evaluated, Pareto-filtered (drop programs dominated on both score and cost), and the loop
repeats. Because feedback is grounded in actual traces, GEPA can fix failure *modes*, not just
tweak wording.

```mermaid
flowchart TD
    A["Parent program"] --> B["Run on mini-batch, collect traces"]
    B --> C["Reflection LLM: why did this fail?"]
    C --> D["Propose improved offspring programs"]
    D --> E["Evaluate candidates"]
    E --> F["Pareto filter: drop dominated programs"]
    F --> G{Iterations left?}
    G -- yes --> B
    G -- no --> H["Best Pareto program"]
```

**Key params:** `metric` (a `GEPAFeedbackMetric` returning feedback text, not just a score),
`reflection_lm` (**required** in dspy 3.3 — the model that writes failure feedback),
`auto` presets, `max_full_evals` / `max_metric_calls` budgets (mutually exclusive),
`reflection_minibatch_size`.

**Budget:** the most expensive prompt optimizer here — reflection calls plus full evaluations;
`auto="light"` exists precisely to bound this.

**When to use:** hard tasks with identifiable failure modes and budget to spend. 🚧 *Not
executed here — the complete step-by-step implementation is **Notebook 12**
(`12_gepa_deep_dive.ipynb`); Notebook 5 adds the MIPROv2 side-by-side comparison).*

In [15]:
# 🚧 SHOWN FOR COMPLETENESS — NOT EXECUTED (see Notebook 5 for a full live run)
def gepa_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    """dspy 3.3 GEPA metrics take the extended 5-arg signature.

    Returning a plain float works; richer metrics return feedback text that GEPA
    uses as evolutionary pressure (see dspy.teleprompt.gepa.GEPAFeedbackMetric).
    """
    return answer_metric(gold, pred, trace)


# Budget knobs are mutually exclusive: pick ONE of auto / max_full_evals / max_metric_calls.
# reflection_lm is REQUIRED in dspy 3.3 — a strong model writes the failure feedback.
optimizer = dspy.GEPA(
    metric=gepa_metric,
    auto="light",
    reflection_lm=dspy.LM("openai/gpt-4o", temperature=1.0),
)  # noqa: F841
# compiled = optimizer.compile(baseline, trainset=trainset)

## 14. `BootstrapFinetune` — Collect Traces, Then Fine-Tune the LM

**What it is:** the first weight-turning optimizer. Instead of (only) editing
prompts, it **fine-tunes the language model** on traces of your program running correctly.

**How it works:** (1) bootstrap the trainset and keep metric-passing traces — input, signature,
ideal output; (2) serialize them into a fine-tuning dataset and launch a **hosted fine-tuning
job** (e.g. OpenAI fine-tuning API); (3) when training finishes, the compiled program pairs
your student with the fine-tuned LM. Prompt optimization changes what you *ask*; this changes
what the model *knows*.

```mermaid
flowchart LR
    A["Teacher/student runs trainset; metric keeps good traces"] --> B["Traces → fine-tuning dataset (JSONL)"]
    B --> C["Launch hosted fine-tune job"]
    C --> D["Fine-tuned LM"]
    D --> E["Compiled program: student + fine-tuned LM"]
```

**Key params:** `metric`, `multitask` (merge all predictors into one training task),
`train_kwargs` / `adapter` (per-LM training config).

**Budget:** bootstrap calls + a real (paid, minutes-to-hours) training job — this cannot be
demoed inside a course notebook. 🚧 *Cell shown, not executed.*
**When to use:** stable tasks with plenty of traces, where prompt optimization has plateaued
or prompts must stay short. **Avoid when** you need quick iteration loops.

In [16]:
# 🚧 SHOWN FOR COMPLETENESS — NOT EXECUTED: launches a paid fine-tuning job
optimizer = dspy.BootstrapFinetune(metric=answer_metric)  # noqa: F841
# compiled = optimizer.compile(baseline, trainset=trainset)
# Then point your program at the fine-tuned LM, e.g.:
# dspy.configure(lm=dspy.LM("openai/finetune-ft-..."))

## 15. `BetterTogether` — Prompts + Weights, Jointly

**What it is:** combines prompt optimization and fine-tuning *in one compile call*:
prompts and model weights improve **together**, which beats either alone (that's the finding
the optimizer is named after).

**How it works:** two stages under one object. Stage 1 runs a prompt optimizer of your choice
(typically `MIPROv2`) to get strong instructions/demos; stage 2 runs `BootstrapFinetune` on the
same task so the LM learns the task distribution while the prompts keep the behavior
inspectable. You pass sub-optimizers as constructor kwargs; `strategy="p -> w -> p"` controls
the order.

```mermaid
flowchart TD
    A["Stage 1: MIPROv2 prompt search"] --> B["Optimized prompts"]
    B --> C["Stage 2: BootstrapFinetune on same task"]
    C --> D["Fine-tuned LM"]
    B --> E["Final program: prompts + weights, jointly"]
    D --> E
```

**Key params:** `metric`, plus named sub-optimizers as constructor kwargs — keys become stage names in the `strategy` string (convention: `p=` the prompt stage, `w=` the weight stage, e.g.
`BetterTogether(metric=..., p=dspy.MIPROv2(...), w=dspy.BootstrapFinetune(...))`).

**Budget:** the sum of both stages — MIPROv2-scale search **plus** a training job. 🚧 *Cell
shown, not executed.* **When to use:** maximum-quality pipelines on stable tasks.
**Avoid when** either budget or iteration speed is constrained — use the stages independently.

In [17]:
# 🚧 SHOWN FOR COMPLETENESS — NOT EXECUTED: full search budget + training job
optimizer = dspy.BetterTogether(
    metric=answer_metric,
    p=dspy.MIPROv2(metric=answer_metric, auto="medium"),  # prompt stage
    w=dspy.BootstrapFinetune(metric=answer_metric),  # weight stage
)  # noqa: F841
# compiled = optimizer.compile(baseline, trainset=trainset, valset=devset)

## Cheat-Sheet: Which Optimizer When?

| Optimizer | Lever | Optimizes | LM budget (small task) | Reach for it when… |
|---|---|---|---|---|
| `LabeledFewShot` | demos | prompt | 0 calls | you have labels and want the cheapest boost |
| `KNN` | retrieval | — | ~2 embedding calls | per-input neighbor lookup |
| `KNNFewShot` | demos + retrieval | prompt | bootstrap **per query** | heterogeneous trainsets, adaptive demos |
| `BootstrapFewShot` | demos | prompt | ~1/example | default first optimizer, no labels needed |
| `Ensemble` | combine | predictions | member inference × N | accuracy > latency/cost |
| `InferRules` | instructions | prompt | bootstrap + candidates × eval | style/format conventions in demos |
| `BootstrapRS` (= `BootstrapFewShotWithRandomSearch`) | demos | prompt | candidates × (bootstrap + eval) | squeezing few-shot performance |
| `COPRO` | instructions | prompt | breadth × depth × eval | wording matters, no demos |
| `SIMBA` | instructions + demos | prompt | steps × (batch + reflection) | mid-size trainsets, clustered failures |
| `MIPROv2` | instructions × demos | prompt | tens–hundreds | serious search when BootstrapFewShot plateaus |
| `GEPA` | instructions | prompt | highest (reflection + eval) | hard tasks with identifiable failure modes |
| `BootstrapFinetune` | weights | LM | bootstrap + training job | prompts plateaued, stable task |
| `BetterTogether` | prompts + weights | both | MIPROv2 + training job | maximum quality on stable tasks |

**Where to go next:**
- Notebook 2 — `BootstrapFewShot` vs `MIPROv2` with devset evaluation and saved programs
- Notebook 12 — `GEPA` complete implementation (deep dive with real Pareto results)
- Notebook 5 — `GEPA` vs MIPROv2 conceptual comparison
- Notebook 7 — the module catalog: what these optimizers are improving